# Submit 29 - ¿Suma el lag 3? (solo para ver en el público, no hay forma de validarlo)

Mismo modelo que el submit 16 (10 trials z005, target juntar, train 202103-202106) con **1 semilla (270131)**, dos archivos top 12.000:
- `29_conlag3_s270131_12000.csv`: lags/deltas de orden 1, 2 **y 3**.
- `29_sinlag3_s270131_12000.csv`: igual al `16_juntar_s270131_12000.csv` (sale de las predicciones guardadas en `probs_16-17/`, no se reentrena).

Ojo: los datos arrancan en 202103 → en el train el lag 3 solo existe para 202106 (toma 202103). Para 202108 toma 202105.
Regla: solo una diferencia de más de ~5 M en el público sería señal (ruido del público ±12 M).
**Los dos archivos van en el mismo submit → no vale como entrega: después reenviar el 28 (o el 21) como último.**

In [1]:
# --- 1. Parámetros ---
import os, time
import numpy as np
import pandas as pd
import lightgbm as lgb

SEMILLA     = 270131
MESES_FINAL = [202103, 202104, 202105, 202106]
MES_PRED    = 202108
N_ENVIO     = 12_000
CARPETA     = "probs_29"
os.makedirs(CARPETA, exist_ok=True)

params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}
elegidos = pd.read_csv("../0_exploracion/z005_top10_final.csv", sep=";", decimal=",",
                       encoding="utf-8-sig", index_col="trial")
list(elegidos.index)

[34, 36, 27, 29, 37, 31, 20, 9, 28, 39]

In [2]:
# --- 2. Datos + lags/deltas de orden 1, 2 y 3 ---
df = pd.read_parquet("../competencia_01.parquet")
no_usar = ["foto_mes", "clase_ternaria"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]
df[cols] = df[cols].astype("float32")          # menos memoria (pasamos de 761 a ~1.060 variables)

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2, 3]:                            # <- la única diferencia con el submit 16: se agrega k = 3
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo
df = df[df["foto_mes"] != 202107]             # julio ya se usó para los lags de agosto; no hace falta más

variables = [c for c in df.columns if c not in no_usar]   # incluye numero_de_cliente
es_final = df["foto_mes"].isin(MESES_FINAL)
futuro = df.loc[df["foto_mes"] == MES_PRED, ["numero_de_cliente"]].copy()
X_futuro = df.loc[futuro.index, variables]
etiqueta = df.loc[es_final, "clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)
print("Variables:", len(variables), "| positivos:", etiqueta.sum())
print("% de NaN del lag 3 por mes:")
print(df.groupby("foto_mes")["ctrx_quarter_lag3"].apply(lambda s: round(s.isna().mean() * 100, 1)))

Variables: 1065 | positivos: 8067
% de NaN del lag 3 por mes:
foto_mes
202103    100.0
202104    100.0
202105    100.0
202106      2.6
202108      2.4
Name: ctrx_quarter_lag3, dtype: float64


In [3]:
# --- 3. Entrenar 10 trials × 1 semilla (retoma si se cortó) ---
datos = lgb.Dataset(df.loc[es_final, variables], label=etiqueta, params={**params_fijos, "seed": SEMILLA})   # max_bin 31 desde el arranque
datos.construct()                              # arma los bines una vez y libera la copia de los datos
inicio_total = time.time()
for trial, r in elegidos.iterrows():
    archivo = f"{CARPETA}/conlag3_t{trial}_s{SEMILLA}.npy"
    if os.path.exists(archivo):
        continue
    p = {**params_fijos, "num_leaves": int(r["params_num_leaves"]),
         "min_sum_hessian_in_leaf": float(r["params_min_sum_hessian_in_leaf"]), "seed": SEMILLA}
    inicio = time.time()
    modelo = lgb.train(p, datos, num_boost_round=int(r["user_attrs_best_iter"]))
    np.save(archivo, modelo.predict(X_futuro))
    with open("29_progreso.txt", "a") as f:
        f.write(f"{time.strftime('%H:%M:%S')} trial {trial}: {(time.time() - inicio) / 60:.1f} min\n")
print(f"Total: {(time.time() - inicio_total) / 60:.0f} min")

Total: 30 min


In [4]:
# --- 4. Dos archivos top 12.000: con lag 3 (nuevo) y sin lag 3 (predicciones del submit 16) ---
tops = {}
for nombre, carpeta, prefijo in [("conlag3", CARPETA, "conlag3"), ("sinlag3", "probs_16-17", "juntar")]:
    probs = [np.load(f"{carpeta}/{prefijo}_t{t}_s{SEMILLA}.npy") for t in elegidos.index]
    f = futuro.copy()
    f["prob"] = np.mean(probs, axis=0)
    top = f.nlargest(N_ENVIO, "prob")["numero_de_cliente"].astype("int64")
    tops[nombre] = set(top)
    archivo = f"29_{nombre}_s{SEMILLA}_{N_ENVIO}.csv"
    top.to_csv(archivo, index=False, header=False)
    with open(archivo) as fh:
        contenido = fh.read()
    print(archivo, "| filas:", len(contenido.splitlines()), "| ¿e+?", "e+" in contenido)

igual_16 = set(pd.read_csv(f"16_juntar_s{SEMILLA}_{N_ENVIO}.csv", header=None)[0]) == tops["sinlag3"]
print("¿sinlag3 es idéntico al archivo del submit 16?", igual_16)
print(f"Coincidencia con lag 3 vs sin lag 3: {len(tops['conlag3'] & tops['sinlag3']) / N_ENVIO * 100:.1f}%")

29_conlag3_s270131_12000.csv | filas: 12000 | ¿e+? False
29_sinlag3_s270131_12000.csv | filas: 12000 | ¿e+? False
¿sinlag3 es idéntico al archivo del submit 16? True
Coincidencia con lag 3 vs sin lag 3: 96.9%


In [5]:
# --- 5. Anotar ---
ruta_res = "../0_exploracion/resultados_experimentos.csv"
res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
res = res[~res["descripcion"].str.startswith("29_lag3")]
res = pd.concat([res, pd.DataFrame([{
    "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "descripcion": f"29_lag3: 2 archivos (con lag 3 / sin lag 3 = submit 16 s{SEMILLA}), 10 trials z005 × 1 semilla, juntar, top {N_ENVIO}",
    "meses_train": str(MESES_FINAL), "mes_valid": MES_PRED, "semilla": str([SEMILLA]),
    "params": str(params_fijos) + " | trials " + str(list(elegidos.index)), "n_enviados": N_ENVIO}])], ignore_index=True)
res["n_enviados"] = res["n_enviados"].astype("Int64")
res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
print("Anotado")

Anotado